In [2]:
import json
import pandas as pd
from pathlib import Path

def aggregate_experiment_results(base_path="runs"):
    all_results = []
    base_dir = Path(base_path)
    
    # Buscamos todos los archivos summary_training.json recursivamente
    summary_files = list(base_dir.glob("**/summary_training.json"))
    
    print(f"🔍 Encontrados {len(summary_files)} archivos de resumen.")

    for file_path in summary_files:
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
            
            # 1. Extraer contexto de la ruta (modelo/experimento/fold)
            # Ejemplo: runs/tf_efficientdet_d0/set1_balanced_subsampled/fold_1_config
            parts = file_path.parts
            # Ajusta los índices según desde dónde lances el script
            model_folder = parts[-3]
            experiment = parts[-2]
            fold_name = parts[-1] 

            # 2. Aplanar la estructura del JSON
            # Extraemos lo más relevante para la comparativa
            row = {
                'model_folder': model_folder,
                'experiment': experiment,
                'fold': file_path.parent.name,
                'architecture': data.get('model_name'),
                'img_size': data.get('model_info_ft', {}).get('img_size'),
                
                # Métricas Principales
                'test_mAP50': data.get('test_mAP50'),
                'test_mAP75': data.get('results_test', {}).get('map_75'),
                'test_mAP_COCO': data.get('results_test', {}).get('map'),
                'best_val_mAP50_FT': data.get('best_val_map50_FT'),
                
                # Inferencia (Velocidad)
                'fps': data.get('inference_stats', {}).get('fps_batch_based'),
                'latency_ms_img': data.get('inference_stats', {}).get('mean_img_time_sec', 0) * 1000,
                
                # Entrenamiento
                'duration': data.get('metadata', {}).get('duration'),
                'params_M': data.get('model_info_ft', {}).get('total_params', 0) / 1e6,
                'early_stop': data.get('early_stop_triggered')
            }
            
            all_results.append(row)
            
        except Exception as e:
            print(f"⚠️ Error procesando {file_path}: {e}")

    # Crear DataFrame
    df = pd.DataFrame(all_results)
    
    # Ordenar por mAP para ver los mejores arriba
    if not df.empty:
        df = df.sort_values(by='test_mAP50', ascending=False).reset_index(drop=True)
        
    return df

# Ejecución
df_results = aggregate_experiment_results("../runs")

# Mostrar top 10
print(df_results.head(10))

# Guardar a CSV para Excel
df_results.to_csv("comparativa_modelos.csv", index=False)

🔍 Encontrados 55 archivos de resumen.
               model_folder     experiment           fold        architecture  \
0          set3_random_full  fold_4_config  fold_4_config  tf_efficientdet_d2   
1          set3_random_full  fold_1_config  fold_1_config  tf_efficientdet_d2   
2        set2_balanced_full  fold_1_config  fold_1_config  tf_efficientdet_d2   
3        set2_balanced_full  fold_2_config  fold_2_config  tf_efficientdet_d2   
4        set2_balanced_full  fold_3_config  fold_3_config  tf_efficientdet_d2   
5          set3_random_full  fold_3_config  fold_3_config  tf_efficientdet_d2   
6        set2_balanced_full  fold_4_config  fold_4_config  tf_efficientdet_d2   
7          set3_random_full  fold_2_config  fold_2_config  tf_efficientdet_d2   
8  set1_balanced_subsampled  fold_1_config  fold_1_config  tf_efficientdet_d2   
9  set1_balanced_subsampled  fold_2_config  fold_2_config  tf_efficientdet_d2   

   img_size  test_mAP50  test_mAP75  test_mAP_COCO  best_val_mAP50_FT 